# 03b. Despliegue del Modelo e Integración RAG Vectorial (Fase de Validación)

## Objetivos de este Notebook
1.  **Carga Dual:** Desplegar el LLM (**Phi-3-mini-4k-instruct**) y el modelo de Embeddings (**BAAI/bge-m3**) en la GPU.
2.  **Conexión Distribuida:** Establecer comunicación con Elasticsearch (alojado en CPU/Valencia) mediante túnel SSH.
3.  **Pipeline RAG Semántico:** Implementar la función de búsqueda vectorial (kNN) con configuración estricta (**Top-k=1**, **Match Perfecto**).
4.  **Validación Técnica:** Realizar las mismas pruebas de cordura que en el BM25 para comparar resultados.

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from sentence_transformers import SentenceTransformer

# 1. Configuración de Hardware
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Hardware detectado: {torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'}")

# 2. Carga del Modelo de Embeddings (NUEVO PARA FASE VECTORIAL)
print("Cargando modelo de Embeddings: BAAI/bge-m3...")
embed_model = SentenceTransformer('BAAI/bge-m3', device=device)

# 3. Selección y Carga del LLM 
MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"
print(f"Cargando modelo LLM: {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",             
    torch_dtype=torch.float16    
)

# 4. Crear Pipeline de Generación
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

print("\n ¡Modelos cargados y listos para inferencia!")

/home/javierruiz/miniconda3/envs/environment/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Hardware detectado: NVIDIA GeForce RTX 4090
Cargando modelo de Embeddings: BAAI/bge-m3...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3647.11it/s, Materializing param=pooler.dense.weight]                               


Cargando modelo LLM: microsoft/Phi-3-mini-4k-instruct...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 195/195 [00:02<00:00, 70.29it/s, Materializing param=model.norm.weight]                              



 ¡Modelos cargados y listos para inferencia!


In [3]:
# 5. Prueba de Inferencia (Sin RAG todavía)
messages = [
    {"role": "user", "content": "¿Podrías explicarme brevemente qué es la inflación económica?"},
]

generation_args = {
    "max_new_tokens": 150,     
    "return_full_text": False, 
    "temperature": 0.1,        
    "do_sample": True,
}

print(" Generando respuesta...")
output = pipe(messages, **generation_args)
print("\n" + "="*50)
print(output[0]['generated_text'][-1]['content'] if isinstance(output[0]['generated_text'], list) else output[0]['generated_text'])
print("="*50)

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Generando respuesta...

 La inflación económica es un fenómeno que se refiere a la tasa a la que el nivel general de precios de bienes y servicios está aumentando, y, por consiguiente, el poder adquisitivo está disminuyendo. En otras palabras, a medida que la inflación aumenta, cada unidad de moneda compra menos bienes y servicios. La inflación puede ser causada por diversos factores, como un aumento en la oferta de dinero, un aumento en los costos de producción, o una disminución en la demanda de bienes y servicios. La inflación moderada es normal en una


In [4]:
from elasticsearch import Elasticsearch

# Conexión a Elasticsearch
es = Elasticsearch("http://127.0.0.1:9250")

try:
    if es.ping():
        print("PING EXITOSO")
        print(f" Conectado a: {es.info()['name']}")
    else:
        print("El servidor no responde. ¿Está el túnel abierto?")
except Exception as e:
    print(f"Error inesperado: {e}")

PING EXITOSO
 Conectado a: valencia


In [5]:
def ask_rag_vectorial(query, top_k=1): 
    """
    Función RAG Vectorial: Recupera 1 noticia (kNN) y genera respuesta basada estrictamente en ella.
    Retorna un diccionario con los datos separados.
    """
    # --- A. BÚSQUEDA VECTORIAL (kNN) ---
    vector_pregunta = embed_model.encode(query).tolist()
    
    query_knn = {
        "field": "vector_texto",
        "query_vector": vector_pregunta,
        "k": top_k,
        "num_candidates": 50
    }
    
    try:
        response = es.search(
            index="noticias_tfg_vectores", 
            knn=query_knn, 
            _source=["title", "body", "date", "source"], 
            size=top_k
        )
        hits = response['hits']['hits']
    except Exception as e:
        return {"error": f"Error en Elasticsearch: {e}"}
    
    if not hits:
        return {"error": " NO ENCONTRADO: No hay ninguna noticia cercana a esta búsqueda."}

    # --- B. EXTRACCIÓN ---
    contexto_unico = ""
    for i, hit in enumerate(hits):
        doc = hit['_source']
        contexto_unico += f"""
        --- NOTICIA {i+1} ---
        TITULO: {doc.get('title')}
        FECHA: {doc.get('date', 'Desconocida')}
        FUENTE: {doc.get('source', 'Desconocida')}
        CONTENIDO: {doc.get('body')}
        \n"""

    # --- C. PROMPT ---
    messages = [
        {"role": "user", "content": f"""
        Eres un sistema de verificación de datos (Fact-Checking). 
        Tu objetivo es responder a la pregunta usando ÚNICAMENTE el texto que te proporciono abajo.

        REGLAS CRÍTICAS:
        1. Si la respuesta no aparece en el texto o no hay informacion que responda a la pregunta, responde exactamente: "No tengo información suficiente en mis archivos".
        2. Formula tu respuesta usando solo los datos del texto. Puedes conectar ideas que estén en la noticia, pero NUNCA uses conocimiento externo ni inventes datos.
        3. No menciones otras noticias que no sean las proporcionadas.
        4. Las respuestas deberán ser cortas y directas.

        ### TEXTO DE REFERENCIA:
        {contexto_unico}
        
        ### PREGUNTA:
        {query}
        """}
    ]

    # --- D. GENERACIÓN ---
    outputs = pipe(
        messages,
        max_new_tokens=256,
        do_sample=False,
        temperature=0.0, 
    )
    
    # --- E. RETORNO ESTRUCTURADO  ---
    respuesta_limpia = outputs[0]['generated_text'][-1]['content']
    
    return {
        "titulo": [hit['_source'].get('title') for hit in hits],
        "contenido": [hit['_source'].get('body') for hit in hits],
        "fecha": [hit['_source'].get('date', 'Desconocida') for hit in hits],
        "fuente": [hit['_source'].get('source', 'Desconocida') for hit in hits],
        "respuesta_rag": respuesta_limpia
    }


In [6]:
pregunta_tfg = "¿Cuánto dinero aproximado le quedó limpio a Carlos Alcaraz por ganar su sexto Grand Slam tras pagar los impuestos federales y de Nueva York?"
print(f" PREGUNTA: {pregunta_tfg}\n")

print("--- RESPUESTA SLM (SIN RAG) ---")
res_base = pipe([{"role": "user", "content": pregunta_tfg}], max_new_tokens=150)
print(res_base[0]['generated_text'][-1]['content']) 

print("\n")

datos = ask_rag_vectorial(pregunta_tfg, top_k=2)
print("--- RESPUESTA SISTEMA RAG VECTORIAL ---")
if "error" in datos:
    print(datos["error"])
else:
    print(f"NOTICIA: {datos['titulo']}")
    print(f"TEXTO: {datos['contenido']} (...)")
    print(f"RESPUESTA: {datos['respuesta_rag']}")

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 PREGUNTA: ¿Cuánto dinero aproximado le quedó limpio a Carlos Alcaraz por ganar su sexto Grand Slam tras pagar los impuestos federales y de Nueva York?

--- RESPUESTA SLM (SIN RAG) ---
 Para estimar cuánto dinero le quedó a Carlos Alcaraz después de ganar su sexto Grand Slam, necesitamos considerar el premio principal del torneo y los impuestos federales y de Nueva York que deben pagarse.


El premio principal del US Open, en el que Carlos Alcaraz ganó su sexto Grand Slam, es de aproximadamente 2.5 millones de dólares. Sin embargo, este no es el monto final que Carlos recibiría, ya que se deben deducir impuestos.


La tasa impositiva sobre el premio principal del Grand Slam varía según la nacional




The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- RESPUESTA SISTEMA RAG VECTORIAL ---
NOTICIA: ['Las millonarias ganancias que alcanzó Carlos Alcaraz tras conquistar su sexto Grand Slam', 'Cuánto dinero en metálico se lleva Carlos Alcaraz por ganar el US Open']
TEXTO: ['El domingo por la tarde, Carlos Alcaraz conquistó el US Open tras derrotar al italiano Jannik Sinner en un partido que quedará en la memoria del tenis. La victoria devolvió al español el número uno del mundo, posición que no ocupaba desde septiembre de 2023, y lo convirtió en seis veces campeón de Grand Slam.   El título neoyorquino le dejó un cheque de 5 millones, cifra que elevó sus ingresos de la temporada a 15.6 millones de dólares y sus ganancias profesionales a casi 54.5 millones. Con ello, Alcaraz se afianza en el sexto lugar histórico del tenis en premios acumulados y se acerca a Alexander Zverev, quien tras su eliminación en tercera ronda se mantiene apenas por delante en la clasificación económica de la ATP. En términos de la temporada 2025, lo que parece

In [8]:
import os
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import warnings
import transformers

# 0. Modo silencioso para evitar los avisos rojos molestos
transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore")

# 1. Configurar el Juez (Llama 3 8B vía Groq)
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    print("ADVERTENCIA: No se encontró GROQ_API_KEY en el entorno .env.")

# 1. Configurar el Juez (Llama 3.1 8B vía Groq)
llm_juez = ChatGroq(
    #model="llama-3.1-8b-instant",
    model="openai/gpt-oss-120b", 
    temperature=0, 
    api_key=GROQ_API_KEY
)

prompt_evaluacion = ChatPromptTemplate.from_messages([
    ("system", """Eres un Juez de Veracidad para un sistema RAG. 
    Tu única misión es validar si la 'Respuesta Generada' coincide con la 'Respuesta Esperada'.

    REGLAS DE ORO:
    1. La respuesta generada puede ser CORRECTA aunque este redactada de forma diferente, siempre que transmita el mismo dato o información esencial que la respuesta esperada.
    2. No uses tu propio conocimiento externo para validar la respuesta, ciñete a la comparación entre la respuesta esperada y la generada.
    3. Asegurate de que el ambas respuestas significan lo mismo de cara a poner CORRECTO

    Responde ÚNICAMENTE con la palabra "CORRECTO" o "INCORRECTO"."""),
    ("human", "Respuesta Esperada:\n{esperada}\n\nRespuesta Generada por el RAG:\n{generada}")
])

juez_chain = prompt_evaluacion | llm_juez | StrOutputParser()

# 2. Cargar el CSV con la batería de preguntas
df_preguntas = pd.read_csv("bateria_preguntas_tfg.csv")
resultados_vectorial = []

print(f"Iniciando evaluación vectorial masiva para {len(df_preguntas)} preguntas...")
print("Usando RAG Vectorial (top_k=2) y Juez Llama-3-8B...\n")

# 3. Bucle de evaluación
# 3. Bucle de evaluación
for index, row in tqdm(df_preguntas.iterrows(), total=len(df_preguntas)):
    pregunta = row['pregunta']
    esperada = row['respuesta_esperada']
    
    # ¡Cambiado a top_k=2 como querías!
    datos = ask_rag_vectorial(pregunta, top_k=2) 
    resp_generada = datos.get('respuesta_rag', datos.get('error', 'Error desconocido'))
    
    # El juez de Groq evalúa
    evaluacion = juez_chain.invoke({"esperada": esperada, "generada": resp_generada}).strip().upper()
    
    if "INCORRECTO" in evaluacion: 
        evaluacion = "INCORRECTO"
    elif "CORRECTO" in evaluacion: 
        evaluacion = "CORRECTO"
    else:
        evaluacion = "INCORRECTO" # Por si el juez se vuelve loco y responde otra cosa
    
    # Guardamos los resultados
    resultados_vectorial.append({
        "Pregunta": pregunta,
        "Esperada": esperada,
        "Respuesta_Generada": resp_generada,
        "Evaluacion": evaluacion
    })

# 4. Guardar y mostrar resultados
df_res_vect = pd.DataFrame(resultados_vectorial)
df_res_vect.to_csv("resultados_solo_vectorial.csv", index=False)

aciertos = (df_res_vect['Evaluacion'] == 'CORRECTO').sum()
total = len(df_res_vect)
porcentaje = (aciertos / total) * 100 if total > 0 else 0

print(f"\nEvaluación terminada. Resultados guardados en 'resultados_solo_vectorial.csv'.")
print(f"RENDIMIENTO DEL RAG VECTORIAL: {aciertos}/{total} Aciertos ({porcentaje:.1f}%)")

# Mostramos las primeras 5 filas para un vistazo rápido
display(df_res_vect[['Pregunta', 'Respuesta_Generada', 'Evaluacion']].head())

Iniciando evaluación vectorial masiva para 80 preguntas...
Usando RAG Vectorial (top_k=2) y Juez Llama-3-8B...



  0%|          | 0/80 [00:00<?, ?it/s]

100%|██████████| 80/80 [03:47<00:00,  2.85s/it]


Evaluación terminada. Resultados guardados en 'resultados_solo_vectorial.csv'.
RENDIMIENTO DEL RAG VECTORIAL: 71/80 Aciertos (88.8%)


,Pregunta,Respuesta_Generada,Evaluacion
0,¿A cuanto llego a subir Trump los aranceles a ...,145%,CORRECTO
1,¿Cuánto dinero perdió Elon Musk tras el anunci...,"Según el texto, Elon Musk perdió 17.800 millo...",INCORRECTO
2,¿Qué opinó el Rey de España sobre los arancele...,No hay información en el texto proporcionado ...,CORRECTO
3,¿Qué declaraciones hizo Elon Musk sobre los ar...,No tengo información suficiente en mis archivos.,CORRECTO
4,¿Qué relación quiere Elon Musk entre Estados U...,Según las declaraciones de Elon Musk en Itali...,CORRECTO
